In [24]:
import pandas as pd
from sklearn.model_selection import train_test_split

# identical 80:20 split from 02
df = pd.read_csv('insurance_cleaned.csv')
X = df.drop(columns=['charges'])
y = df['charges']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=69
)

3.1 Transformación polynómica
    polynomial transformation(try different grados) (en train)

In [25]:
from sklearn.preprocessing import PolynomialFeatures

degrees = [1, 2, 3, 4]

for d in degrees:
    poly = PolynomialFeatures(degree=d, include_bias=False)
    X_train_poly = poly.fit_transform(X_train)



3.2 Entrenamiento del modelo
    train linear regression model over transformed variables

In [26]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

# 5-fold cross validation strictly on training partition
kf = KFold(n_splits=5, shuffle=True, random_state=69)
cv_poly_results = {}

for d in degrees:
    pipe_poly = Pipeline(
        [
            ('poly', PolynomialFeatures(degree=d, include_bias=False)),
            ('scaler', StandardScaler()),
            ('reg', LinearRegression()),
        ]
    )

    scores = cross_validate(
        pipe_poly,
        X_train,
        y_train,
        cv=kf,
        scoring=['neg_root_mean_squared_error', 'r2'],
        return_train_score=True,
    )

    cv_poly_results[d] = {
        'train_rmse': -scores['train_neg_root_mean_squared_error'].mean(),
        'val_rmse': -scores['test_neg_root_mean_squared_error'].mean(),
        'train_r2': scores['train_r2'].mean(),
        'val_r2': scores['test_r2'].mean(),
    }

    print(
        f"Degree {d} | Train RMSE: {cv_poly_results[d]['train_rmse']:.4f} | "
        f"Val RMSE: {cv_poly_results[d]['val_rmse']:.4f} | Val R²: {cv_poly_results[d]['val_r2']:.4f}"
    )

Degree 1 | Train RMSE: 5891.7876 | Val RMSE: 5923.0506 | Val R²: 0.7553
Degree 2 | Train RMSE: 4567.1854 | Val RMSE: 4682.0172 | Val R²: 0.8463
Degree 3 | Train RMSE: 4457.6224 | Val RMSE: 4707.6827 | Val R²: 0.8447
Degree 4 | Train RMSE: 4331.9315 | Val RMSE: 5098.7025 | Val R²: 0.8168


3.3 Regularización (optional -> do last)


In [27]:
import warnings
import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

# Silence ConvergenceWarning and UserWarning (for alpha near zero)
warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore', category=UserWarning)

kf = KFold(n_splits=5, shuffle=True, random_state=69)

# Dense search grid around the optima without exact 0.0
alpha_candidates = np.unique(
    np.concatenate(
        [
            np.logspace(-5, -1, 10),  # Very small alphas near zero
            np.linspace(1.0, 35.0, 69),  # Fine step size (0.5) around optimal range
        ]
    )
)

param_grid = {'lasso__alpha': alpha_candidates}

lasso_summary = []

for d in [1, 2, 3]:
    pipe_lasso = Pipeline(
        [
            ('poly', PolynomialFeatures(degree=d, include_bias=False)),
            ('scaler', StandardScaler()),
            ('lasso', Lasso(max_iter=5000, tol=1e-3, random_state=69)),
        ]
    )

    grid = GridSearchCV(
        pipe_lasso,
        param_grid,
        cv=kf,
        scoring='neg_root_mean_squared_error',
        return_train_score=True,
        n_jobs=1,
    )

    grid.fit(X_train, y_train)

    best_idx = grid.best_index_
    train_rmse = -grid.cv_results_['mean_train_score'][best_idx]
    val_rmse = -grid.best_score_
    best_alpha = grid.best_params_['lasso__alpha']

    lasso_summary.append(
        {
            'Degree': d,
            'Optimal Lambda (alpha)': best_alpha,
            'CV Train RMSE': train_rmse,
            'CV Val RMSE': val_rmse,
        }
    )

    print(
        f"Degree {d} | Optimal Alpha: {best_alpha:10.4f} | "
        f"CV Train RMSE: {train_rmse:.4f} | CV Val RMSE: {val_rmse:.4f}"
    )

df_lasso_results = pd.DataFrame(lasso_summary)

Degree 1 | Optimal Alpha:     0.0000 | CV Train RMSE: 5891.7876 | CV Val RMSE: 5923.0506
Degree 2 | Optimal Alpha:    17.5000 | CV Train RMSE: 4588.3219 | CV Val RMSE: 4654.4112
Degree 3 | Optimal Alpha:    12.0000 | CV Train RMSE: 4536.8196 | CV Val RMSE: 4680.6568


we will use Degree 2 with alpha 17.5